# NB02 — BTCS: Budgeted Temporal Case Segmentation
**Andre da Costa Silva | ITA | 2026**

Algoritmo proposto para ICDM 2026:
1. **Grafo auxiliar Lk**: nos = top-k arestas, conexoes por proximidade temporal + estrutural
2. **Leiden**: particionar Lk em comunidades (= casos)
3. **Budget B**: limitar arestas induzidas por caso a no maximo B

Executar: celulas 1->2->3->4->5->6->7->8->9 (ou Runtime->Run all)

In [ ]:
# CELULA 1 - Setup: imports, paths, verificacao
import os, sys, subprocess, time, math, contextlib
from pathlib import Path
from collections import defaultdict

def _pip(*pkgs):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + list(pkgs))

_pip('numpy', 'pandas', 'scikit-learn', 'matplotlib', 'psutil', 'tqdm',
     'python-igraph', 'leidenalg')
try:
    import torch
except ImportError:
    _pip('torch', '--index-url', 'https://download.pytorch.org/whl/cu121')
    import torch

import numpy as np
import pandas as pd
import psutil
import igraph as ig
import leidenalg
import matplotlib.pyplot as plt
from IPython.display import display

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'torch {torch.__version__} | igraph {ig.__version__} | leidenalg {leidenalg.version}')
print(f'device: {DEVICE}')

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
    except Exception as e:
        print(f'[WARN] Drive: {e}')

BASE = Path('/content/drive/MyDrive') if IN_COLAB else Path('.').resolve()

AML100K_BASE  = BASE / 'DatasetDissertacao/IBM_TRANSACTION_AML/AMLSIMFULL/AML100k'
AML100K_ARTIF = AML100K_BASE / 'artifacts'
AML100K_PROBS = AML100K_BASE / 'results/probs_v4'
AML100K_MODEL = 'SAGE'
AML100K_SEED  = 42

AML1M_BASE    = BASE / 'DatasetDissertacao/IBM_TRANSACTION_AML/AMLSIMFULL/AML1M'
AML1M_ARTIF   = AML1M_BASE / 'artifacts'
AML1M_PROBS   = AML1M_BASE / 'results_aml1m_graphsage_only/probs_v4'
AML1M_MODEL   = 'GraphSAGE'
AML1M_SEED    = 44

BTCS_OUT = BASE / 'GrafosGNN/results/nb02_btcs'
BTCS_OUT.mkdir(parents=True, exist_ok=True)
NB01_OUT = BASE / 'GrafosGNN/results/nb01_baselines'

print('\n=== Verificacao ===')
for label, path in [
    ('AML100k cache', AML100K_ARTIF / 'edge_data_v4_clean.pt'),
    ('AML100k probs', AML100K_PROBS / f'{AML100K_MODEL}_seed{AML100K_SEED}_test.npz'),
    ('AML1M   cache', AML1M_ARTIF   / 'edge_data_v4_clean.pt'),
    ('AML1M   probs', AML1M_PROBS   / f'{AML1M_MODEL}_seed{AML1M_SEED}_test.npz'),
    ('nb01 results', NB01_OUT / 'b0_b1_b2_b3_results.csv'),
]:
    print(f"  {'ok' if path.exists() else 'MISS'}  {label}")

In [ ]:
# CELULA 2 - Funcoes base (reutilizadas do nb01)

@contextlib.contextmanager
def measure_resources():
    proc = psutil.Process(os.getpid())
    m0 = proc.memory_info().rss / 1024**2
    t0 = time.perf_counter()
    r  = {}
    yield r
    r['time_s'] = time.perf_counter() - t0
    r['ram_mb']  = proc.memory_info().rss / 1024**2 - m0

def evaluate_cases(cases, gt, ranked, k):
    y_full = np.asarray(gt['y_full'], dtype=int)
    y_test = np.asarray(ranked['y'],  dtype=int)
    pos_te = int(y_test.sum())
    K      = max(1, int(math.ceil(k * len(y_test))))
    nan_row = {m: np.nan for m in [
        'n_cases','coverage','purity_induced','cr_at_k','recall_in_cases',
        'edges_per_case_median','edges_per_case_p95','edges_per_case_max',
        'e_ind_total','e_ind_over_ek','ocr_b100','ocr_b500','ocr_b1000']}
    if not cases:
        return nan_row
    ind_sizes = np.array([len(c['induced_edges']) for c in cases])
    non_empty = [c['induced_edges'] for c in cases if c['induced_edges']]
    if non_empty:
        all_ind = np.unique(np.concatenate(
            [np.asarray(x, dtype=np.int64) for x in non_empty]))
    else:
        all_ind = np.array([], dtype=np.int64)
    pos_ind  = int(y_full[all_ind].sum()) if len(all_ind) > 0 else 0
    coverage = float(pos_ind / max(pos_te, 1))
    purity   = float(pos_ind / max(int(ind_sizes.sum()), 1))
    pos_sel  = sum(int(y_test[c['seed_edges']].sum()) for c in cases if c['seed_edges'])
    recall   = float(pos_sel / max(pos_te, 1))
    cr_at_k  = float(recall / max(coverage, 1e-12)) if coverage > 0 else np.nan
    return {'n_cases':len(cases), 'coverage':coverage, 'purity_induced':purity,
            'cr_at_k':cr_at_k, 'recall_in_cases':recall,
            'edges_per_case_median':float(np.median(ind_sizes)),
            'edges_per_case_p95':float(np.percentile(ind_sizes, 95)),
            'edges_per_case_max':float(ind_sizes.max()),
            'e_ind_total':int(ind_sizes.sum()),
            'e_ind_over_ek':float(ind_sizes.sum() / max(K, 1)),
            'ocr_b100':float((ind_sizes>100).mean()),
            'ocr_b500':float((ind_sizes>500).mean()),
            'ocr_b1000':float((ind_sizes>1000).mean())}

def load_dataset_artifacts(artif_dir, probs_dir, model, seed, dataset_name=''):
    artif_dir = Path(artif_dir); probs_dir = Path(probs_dir)
    npz  = np.load(probs_dir / f'{model}_seed{seed}_test.npz')
    p_te = np.asarray(npz['p'], dtype=float)
    y_te = np.asarray(npz['y'], dtype=int)
    print(f'[{dataset_name}] {len(p_te):,} arestas teste, {y_te.sum():,} pos ({100*y_te.mean():.2f}%)')
    cache  = torch.load(artif_dir/'edge_data_v4_clean.pt', map_location='cpu', weights_only=False)
    ei_all = cache['ei_all_cpu']
    te_idx = cache['te_idx']
    if isinstance(te_idx, torch.Tensor): te_idx = te_idx.numpy()
    if isinstance(ei_all, torch.Tensor): ei_all = ei_all.numpy()
    src_te = ei_all[0, te_idx]; dst_te = ei_all[1, te_idx]
    assert len(p_te)==len(src_te), f'Mismatch: {len(p_te)} vs {len(src_te)}'
    print(f'[{dataset_name}] Cache: {ei_all.shape[1]:,} totais, {len(te_idx):,} teste')
    return ({'scores':p_te,'y':y_te,'src':src_te,'dst':dst_te},
            {'src':src_te,'dst':dst_te},
            {'y_full':y_te}, te_idx, ei_all)

def load_timestamps_from_csv(data_dir, te_idx, ei_all):
    data_dir = Path(data_dir)
    candidates = ['transactions.csv','transaction.csv',
                  'hi-large_trans.csv','hi-medium_trans.csv','hi-small_trans.csv',
                  'li-large_trans.csv','li-medium_trans.csv','li-small_trans.csv']
    csv_path = next((list(data_dir.rglob(c))[0] for c in candidates
                     if list(data_dir.rglob(c))), None)
    if csv_path is None:
        raise FileNotFoundError(f'CSV nao encontrado em {data_dir}')
    print(f'  CSV: {csv_path.name}')
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip().str.lower()
    time_col = next((c for c in ['timestamp','time','date','datetime','step'] if c in df.columns), None)
    src_col  = next((c for c in ['sender_account_id','src','source','from','sender','src_id'] if c in df.columns), None)
    dst_col  = next((c for c in ['receiver_account_id','dst','dest','to','receiver','dst_id'] if c in df.columns), None)
    if time_col is None:
        raise KeyError(f'Coluna tempo nao encontrada. Cols: {list(df.columns)}')
    print(f'  Colunas: time={time_col!r}  src={src_col!r}  dst={dst_col!r}')
    if pd.api.types.is_numeric_dtype(df[time_col]):
        ts_raw = pd.to_numeric(df[time_col], errors='coerce').fillna(0).astype(np.int64).values
    else:
        ts_raw = (pd.to_datetime(df[time_col], errors='coerce')
                    .fillna(pd.Timestamp('1970-01-01')).astype(np.int64)).values
    order = np.argsort(ts_raw, kind='stable')
    ts_sort = ts_raw[order]
    if src_col and dst_col:
        mask = df[src_col].astype(str).values[order] != df[dst_col].astype(str).values[order]
        ts_clean = ts_sort[mask]
        n_loops = int((~mask).sum())
        if n_loops > 0: print(f'  Self-loops removidos: {n_loops}')
    else:
        ts_clean = ts_sort
    n_csv, n_cache = len(ts_clean), ei_all.shape[1]
    if n_csv != n_cache:
        raise AssertionError(f'Mismatch: {n_csv} ts vs {n_cache} arestas')
    ts_test = ts_clean[te_idx]
    print(f'  Timestamps: [{ts_test.min()}, {ts_test.max()}]')
    return ts_test

print('Funcoes base definidas.')

In [ ]:
# CELULA 3 - BTCS: Grafo auxiliar Lk + Leiden + Budget
#
# Etapa 1: Construir Lk
#   - Nos de Lk = top-k arestas (indices 0..K-1)
#   - Aresta em Lk entre e_i e e_j se:
#     (a) compartilham pelo menos um no no grafo original
#     (b) |ts[e_i] - ts[e_j]| <= delta_L
#   - Otimizacao: janela deslizante com early termination O(K * d_avg)
#
# Etapa 2: Leiden
#   - Particiona Lk em comunidades (RBConfigurationVertexPartition)
#   - resolution (gamma) controla granularidade: maior = mais casos menores
#
# Etapa 3: Budget B
#   - Para cada caso: encontra arestas induzidas (vetorizado O(E))
#   - Se |induced| > B: manter as B com maior score do modelo
#   - Garante OCR(B) = 0 por construcao

def build_Lk(src_sel, dst_sel, ts_sel, delta_L, max_hub_edges=500):
    """Constroi grafo auxiliar Lk. Retorna lista de arestas (i,j).
    max_hub_edges: limita conexoes para nos com muitas arestas top-k."""
    K = len(src_sel)
    node_map = defaultdict(list)
    for i in range(K):
        node_map[int(src_sel[i])].append((i, int(ts_sel[i])))
        node_map[int(dst_sel[i])].append((i, int(ts_sel[i])))
    
    lk_edges = set()
    n_hubs_capped = 0
    for node, elist in node_map.items():
        n = len(elist)
        if n < 2:
            continue
        if n > max_hub_edges:
            n_hubs_capped += 1
            elist.sort(key=lambda x: x[1])
            elist = elist[:max_hub_edges]
        else:
            elist.sort(key=lambda x: x[1])
        
        for i in range(len(elist)):
            ei, ti = elist[i]
            for j in range(i+1, len(elist)):
                ej, tj = elist[j]
                if tj - ti > delta_L:
                    break
                if ei != ej:
                    pair = (min(ei, ej), max(ei, ej))
                    lk_edges.add(pair)
    
    if n_hubs_capped > 0:
        print(f'  Lk: {n_hubs_capped} nos capped em {max_hub_edges}')
    return list(lk_edges)


def leiden_partition(K, lk_edges, resolution=1.0, seed=42):
    """Particiona Lk com Leiden. Retorna membership array."""
    g = ig.Graph(n=K, edges=lk_edges, directed=False)
    g.simplify()
    partition = leidenalg.find_partition(
        g, leidenalg.RBConfigurationVertexPartition,
        resolution_parameter=resolution,
        seed=seed)
    membership = np.array(partition.membership, dtype=np.int64)
    n_comms = len(set(membership))
    sizes = np.bincount(membership)
    print(f'  Leiden: {n_comms} comunidades | '
          f'median={np.median(sizes):.0f} mean={sizes.mean():.1f} '
          f'max={sizes.max()} | Q={partition.quality():.4f}')
    return membership


def build_btcs_cases(ranked, full, k=0.01, delta_L=7, resolution=1.0,
                     budget_B=100, min_nodes=3, seed=42):
    """Pipeline completo BTCS: Lk -> Leiden -> Induced Edges -> Budget."""
    scores = np.asarray(ranked['scores'], dtype=float)
    src    = np.asarray(ranked['src'],    dtype=np.int64)
    dst    = np.asarray(ranked['dst'],    dtype=np.int64)
    ts     = np.asarray(ranked['timestamps'], dtype=np.int64)
    sf     = np.asarray(full['src'],      dtype=np.int64)
    df_    = np.asarray(full['dst'],      dtype=np.int64)
    K      = max(1, int(math.ceil(k * len(scores))))
    sel    = np.argsort(-scores)[:K]
    
    src_sel = src[sel]
    dst_sel = dst[sel]
    ts_sel  = ts[sel]
    
    print(f'  K={K:,} arestas selecionadas')
    
    # --- Etapa 1: Construir Lk ---
    t0 = time.perf_counter()
    lk_edges = build_Lk(src_sel, dst_sel, ts_sel, delta_L)
    t_lk = time.perf_counter() - t0
    print(f'  Lk: {K:,} nos, {len(lk_edges):,} arestas ({t_lk:.1f}s)')
    
    if len(lk_edges) == 0:
        print('  [WARN] Lk sem arestas - cada top-k edge vira caso isolado')
        membership = np.arange(K, dtype=np.int64)
    else:
        # --- Etapa 2: Leiden ---
        t0 = time.perf_counter()
        membership = leiden_partition(K, lk_edges, resolution, seed)
        t_leiden = time.perf_counter() - t0
        print(f'  Leiden: {t_leiden:.1f}s')
    
    # --- Montar casos a partir do membership ---
    max_node = int(max(sf.max(), df_.max(), src_sel.max(), dst_sel.max())) + 1
    n_comms = int(membership.max()) + 1
    
    cases_raw = [{'nodes':set(), 'seed_edges':[], 'induced_edges':[]}
                 for _ in range(n_comms)]
    for i in range(K):
        g = int(membership[i])
        cases_raw[g]['seed_edges'].append(int(sel[i]))
        cases_raw[g]['nodes'].update([int(src_sel[i]), int(dst_sel[i])])
    
    # Filtrar por min_nodes
    cases = [c for c in cases_raw if len(c['nodes']) >= min_nodes]
    if not cases:
        return []
    
    n_filtered = n_comms - len(cases)
    if n_filtered > 0:
        print(f'  {n_filtered} casos removidos (< {min_nodes} nos)')
    
    # gid_of[node] = indice do caso (casos disjuntos por Leiden)
    gid_of = -np.ones(max_node, dtype=np.int64)
    for g, case in enumerate(cases):
        for nid in case['nodes']:
            if nid < max_node:
                gid_of[nid] = g
    
    # --- Etapa 3a: Arestas induzidas (vetorizado O(E)) ---
    t0 = time.perf_counter()
    g_src = np.where(sf < max_node, gid_of[sf], -1)
    g_dst = np.where(df_ < max_node, gid_of[df_], -1)
    mask  = (g_src == g_dst) & (g_src >= 0)
    idx_valid = np.where(mask)[0]
    
    if len(idx_valid) > 0:
        g_valid = g_src[idx_valid]
        order   = np.argsort(g_valid, kind='stable')
        g_sorted = g_valid[order]
        i_sorted = idx_valid[order]
        uq, cnt  = np.unique(g_sorted, return_counts=True)
        for g_id, grp in zip(uq, np.split(i_sorted, np.cumsum(cnt)[:-1])):
            cases[g_id]['induced_edges'] = grp.tolist()
    
    t_ind = time.perf_counter() - t0
    
    # --- Etapa 3b: Aplicar Budget B ---
    n_capped = 0
    if budget_B is not None and budget_B > 0:
        scores_full = np.asarray(ranked['scores'], dtype=float)
        for case in cases:
            if len(case['induced_edges']) > budget_B:
                n_capped += 1
                idx = np.array(case['induced_edges'], dtype=np.int64)
                sc_ind = scores_full[idx]
                top_b = idx[np.argsort(-sc_ind)[:budget_B]]
                case['induced_edges'] = top_b.tolist()
    
    ind_sizes = [len(c['induced_edges']) for c in cases]
    print(f'  Induced: {t_ind:.1f}s | Budget B={budget_B}: '
          f'{n_capped}/{len(cases)} casos capped')
    print(f'  Final: {len(cases)} casos | '
          f'median={np.median(ind_sizes):.0f} max={max(ind_sizes)}')
    
    return cases

print('BTCS (build_Lk + leiden_partition + build_btcs_cases) definido.')

In [ ]:
# CELULA 4 - Carregar dados + timestamps
print('Carregando AML100k...')
ranked_100k, full_100k, gt_100k, te_idx_100k, ei_100k = load_dataset_artifacts(
    AML100K_ARTIF, AML100K_PROBS, AML100K_MODEL, AML100K_SEED, 'AML100k')
ts_100k = load_timestamps_from_csv(AML100K_BASE, te_idx_100k, ei_100k)
ranked_100k['timestamps'] = ts_100k
full_100k['timestamps']   = ts_100k

print('\nCarregando AML1M...')
ranked_1m, full_1m, gt_1m, te_idx_1m, ei_1m = load_dataset_artifacts(
    AML1M_ARTIF, AML1M_PROBS, AML1M_MODEL, AML1M_SEED, 'AML1M')
ts_1m = load_timestamps_from_csv(AML1M_BASE, te_idx_1m, ei_1m)
ranked_1m['timestamps'] = ts_1m
full_1m['timestamps']   = ts_1m

# Carregar resultados do nb01 para comparacao
df_baselines = pd.read_csv(NB01_OUT / 'b0_b1_b2_b3_results.csv')
print(f'\nnb01 baselines: {len(df_baselines)} linhas carregadas')
print('Dados prontos!')

In [ ]:
# CELULA 5 - Hiperparametros BTCS
DELTA_L     = 7       # janela temporal para Lk. Sweep: {3, 7, 14, 30}
RESOLUTIONS = [0.5, 1.0, 2.0, 5.0]  # gamma do Leiden
BUDGET_B    = 100     # B do paper. Sweep: {50, 100, 200, 500, None}
KS          = [0.01, 0.02, 0.05, 0.10]
MIN_NODES   = 3

print(f'delta_L={DELTA_L}  resolutions={RESOLUTIONS}')
print(f'budget_B={BUDGET_B}  k={[f"{int(k*100)}%" for k in KS]}')

In [ ]:
# CELULA 6 - Rodar BTCS em varias configuracoes
DATASETS = [
    ('AML100k', ranked_100k, full_100k, gt_100k),
    ('AML1M',   ranked_1m,   full_1m,   gt_1m),
]
rows = []

for ds, ranked, full, gt in DATASETS:
    print(f'\n{"="*50}  {ds}  {"="*50}')
    for gamma in RESOLUTIONS:
        for k in KS:
            kp = f'{int(k*100)}%'
            print(f'\n  gamma={gamma} k={kp}:')
            
            with measure_resources() as res:
                cases = build_btcs_cases(
                    ranked, full, k=k,
                    delta_L=DELTA_L, resolution=gamma,
                    budget_B=BUDGET_B, min_nodes=MIN_NODES)
                m = evaluate_cases(cases, gt, ranked, k)
            
            row = {'dataset':ds, 'method':f'BTCS_g{gamma}',
                   'k%':kp, 'k_frac':k,
                   'delta_L':DELTA_L, 'resolution':gamma,
                   'budget_B':BUDGET_B,
                   **m, **res}
            rows.append(row)
            print(f"    BTCS(g={gamma}): #casos={m['n_cases']:,} "
                  f"cov={m['coverage']:.3f} pur={m['purity_induced']:.4f} "
                  f"OCR100={m['ocr_b100']:.3f} E/Ek={m['e_ind_over_ek']:.2f}x "
                  f"{res['time_s']:.1f}s")

df_btcs = pd.DataFrame(rows)
print('\nBTCS concluido!')

In [ ]:
# CELULA 7 - Comparacao BTCS vs Baselines
print('=== Selecao do melhor gamma por (dataset, k) ===\n')

for ds in ['AML100k', 'AML1M']:
    print(f'--- {ds} ---')
    dfd = df_btcs[df_btcs['dataset']==ds]
    for kp in ['1%', '5%', '10%']:
        sub = dfd[dfd['k%']==kp].sort_values('coverage', ascending=False)
        best = sub[sub['ocr_b100'] <= 0.01]
        if best.empty:
            best = sub
        w = best.iloc[0]
        print(f'  k={kp}: gamma={w["resolution"]} '
              f'cov={w["coverage"]:.3f} OCR100={w["ocr_b100"]:.3f} '
              f'E/Ek={w["e_ind_over_ek"]:.2f}x')
    print()

# Tabela comparativa: B0, B1, BTCS
print('\n=== Tabela Comparativa Final ===')
for ds in ['AML100k', 'AML1M']:
    print(f'\n{"="*40} {ds} {"="*40}')
    bl = df_baselines[df_baselines['dataset']==ds]
    bt_all = df_btcs[df_btcs['dataset']==ds]
    
    comp_rows = []
    for kp in ['1%', '2%', '5%', '10%']:
        # B0
        b0r = bl[(bl['method']=='B0_WCC') & (bl['k%']==kp)]
        if not b0r.empty:
            r = b0r.iloc[0]
            comp_rows.append({'method':'B0_WCC','k%':kp,
                'n_cases':r['n_cases'],'coverage':r['coverage'],
                'purity':r['purity_induced'],'ocr_b100':r['ocr_b100'],
                'e_ind_over_ek':r['e_ind_over_ek'],'time_s':r['time_s']})
        # B1
        b1r = bl[(bl['method']=='B1_TemporalWCC') & (bl['k%']==kp)]
        if not b1r.empty:
            r = b1r.iloc[0]
            comp_rows.append({'method':'B1_Temporal','k%':kp,
                'n_cases':r['n_cases'],'coverage':r['coverage'],
                'purity':r['purity_induced'],'ocr_b100':r['ocr_b100'],
                'e_ind_over_ek':r['e_ind_over_ek'],'time_s':r['time_s']})
        # BTCS best
        sub = bt_all[bt_all['k%']==kp].sort_values('coverage', ascending=False)
        cand = sub[sub['ocr_b100'] <= 0.01]
        if cand.empty:
            cand = sub
        r = cand.iloc[0]
        comp_rows.append({'method':f'BTCS(g={r["resolution"]:.1f})','k%':kp,
            'n_cases':r['n_cases'],'coverage':r['coverage'],
            'purity':r['purity_induced'],'ocr_b100':r['ocr_b100'],
            'e_ind_over_ek':r['e_ind_over_ek'],'time_s':r['time_s']})
    
    df_comp = pd.DataFrame(comp_rows)
    display(df_comp.round(4).pivot_table(
        index='k%', columns='method',
        values=['coverage','ocr_b100','e_ind_over_ek','purity'],
        aggfunc='first').round(4))

In [ ]:
# CELULA 8 - Plots
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for col, ds in enumerate(['AML100k', 'AML1M']):
    bl = df_baselines[df_baselines['dataset']==ds]
    bt = df_btcs[df_btcs['dataset']==ds]
    
    # --- Plot 1: Coverage vs OCR scatter (k=1%) ---
    ax = axes[0, col]
    kp = '1%'
    for method, color, marker, label in [
        ('B0_WCC','#1f77b4','s','B0 WCC'),
        ('B1_TemporalWCC','#ff7f0e','^','B1 Temporal'),
        ('B2_WCC_Ctx','#2ca02c','v','B2 WCC+Ctx'),
        ('B3_HubPruned','#d62728','D','B3 Hub-Pruned')]:
        row = bl[(bl['method']==method) & (bl['k%']==kp)]
        if not row.empty:
            ax.scatter(float(row['ocr_b100']), float(row['coverage']),
                      c=color, marker=marker, s=100, label=label, zorder=5)
    
    for gamma in RESOLUTIONS:
        row = bt[(bt['resolution']==gamma) & (bt['k%']==kp)]
        if not row.empty:
            lbl = f'BTCS g={gamma}' if gamma == RESOLUTIONS[0] else ''
            ax.scatter(float(row['ocr_b100']), float(row['coverage']),
                      c='black', marker='*', s=200, zorder=10, label=lbl)
            ax.annotate(f'g={gamma}',
                       (float(row['ocr_b100'])+0.001, float(row['coverage'])),
                       fontsize=7)
    
    ax.set_xlabel('OCR(B=100)')
    ax.set_ylabel('Coverage')
    ax.set_title(f'{ds} k={kp}: Coverage vs OCR(100)')
    ax.legend(fontsize=7, loc='lower left')
    ax.axhline(y=0.8, color='gray', ls='--', alpha=0.3)
    ax.axvline(x=0.01, color='gray', ls='--', alpha=0.3)
    
    # --- Plot 2: OCR barplot (all k, B0 vs B1 vs BTCS best) ---
    ax = axes[1, col]
    labels_all = ['B0 WCC', 'B1 Temporal', 'BTCS']
    colors_all = ['#1f77b4', '#ff7f0e', '#2ca02c']
    x = np.arange(len(KS))
    w = 0.25
    
    for j, (method, color, label) in enumerate(zip(
            ['B0_WCC', 'B1_TemporalWCC', 'BTCS_best'], colors_all, labels_all)):
        vals = []
        for k_val in KS:
            kp_v = f'{int(k_val*100)}%'
            if method == 'BTCS_best':
                sub = bt[bt['k%']==kp_v].sort_values('coverage', ascending=False)
                cand = sub[sub['ocr_b100'] <= 0.01]
                if cand.empty:
                    cand = sub
                vals.append(float(cand.iloc[0]['ocr_b100']))
            else:
                row = bl[(bl['method']==method) & (bl['k%']==kp_v)]
                vals.append(float(row['ocr_b100']) if not row.empty else 0)
        ax.bar(x + j*w, vals, w, label=label, color=color, alpha=0.85)
    
    ax.set_title(f'{ds} - OCR(B=100): B0 vs B1 vs BTCS')
    ax.set_xticks(x + w)
    ax.set_xticklabels([f'{int(k*100)}%' for k in KS])
    ax.set_ylabel('fracao casos >100 arestas')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(BTCS_OUT / 'btcs_vs_baselines.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot salvo.')

In [ ]:
# CELULA 9 - Export
df_btcs.to_csv(BTCS_OUT / 'btcs_results.csv', index=False)
print(f'CSV: {BTCS_OUT}/btcs_results.csv')

# Combinar com baselines para tabela LaTeX final
cols = ['dataset','method','k%','n_cases','coverage','purity_induced',
        'cr_at_k','e_ind_over_ek','ocr_b100','time_s']
bl_sub = df_baselines[df_baselines['method'].isin(
    ['B0_WCC','B1_TemporalWCC'])][cols]

# Selecionar melhor BTCS por (dataset, k)
btcs_best_rows = []
for ds in ['AML100k','AML1M']:
    for kp in ['1%','2%','5%','10%']:
        sub = df_btcs[(df_btcs['dataset']==ds) & (df_btcs['k%']==kp)]
        sub = sub.sort_values('coverage', ascending=False)
        cand = sub[sub['ocr_b100'] <= 0.01]
        if cand.empty:
            cand = sub
        row = cand.iloc[0].copy()
        row['method'] = f"BTCS(g={row['resolution']:.1f},B={int(row['budget_B'])})"
        btcs_best_rows.append(row)
bt_sub = pd.DataFrame(btcs_best_rows)[cols]

df_final = pd.concat([bl_sub, bt_sub], ignore_index=True)
df_final[cols].round(4).to_latex(
    BTCS_OUT / 'btcs_comparison_table.tex', index=False, escape=False)
print('LaTeX salvo.')

print('\nProgresso:')
print('1.[ok] nb00  2.[ok] nb01  3.[ok] nb02-BTCS')
print('4.[ ] nb03-ablacoes  5.[ ] nb04-multidataset')